# flash_mla smoke test

Build from source: [github.com/vllm-project/FlashMLA](https://github.com/vllm-project/FlashMLA) (the distribution on PyPI is a placeholder).


## Example 1 — the import, the version and the one compiled extension


In [ ]:
import importlib.metadata, os
import torch, flash_mla

print("flash_mla", flash_mla.__version__)
assert importlib.metadata.version("flash_mla").split("+")[0] == flash_mla.__version__, (
    flash_mla.__version__, importlib.metadata.version("flash_mla"))
folder = os.path.dirname(flash_mla.__file__)
extensions = [name for name in os.listdir(folder) if name.startswith("_flashmla_C") and name.endswith(".so")]
assert len(extensions) == 1, extensions  # `import flash_mla` loads exactly this one with torch.ops.load_library
for name in flash_mla.__all__:
    assert hasattr(flash_mla, name), f"flash_mla.__all__ names {name}, which is missing"
# The interface calls the operators the extension registered, under the name vLLM uses.
assert flash_mla.flash_mla_interface.flash_mla_cuda is torch.ops._flashmla_C
print("OK -- the interface and the compiled extension loaded:", extensions[0])


## Example 2 — the operators take the arguments vLLM's FlashMLA interface passes


In [ ]:
import torch, flash_mla

def argument_names(operator):
    (schema,) = torch._C._jit_get_schemas_for_operator(f"_flashmla_C::{operator}")
    return [argument.name for argument in schema.arguments]

# vLLM calls these by position, so the argument lists must be the ones of the commit vLLM names.
sparse_prefill = argument_names("sparse_prefill_fwd")
assert sparse_prefill == ["q", "kv", "indices", "sm_scale", "d_v", "attn_sink", "topk_length", "out_"], sparse_prefill
sparse_decode = argument_names("sparse_decode_fwd")
assert sparse_decode[:5] == ["q", "kv", "indices", "topk_length", "attn_sink"] and len(sparse_decode) == 13, sparse_decode
dense_decode = argument_names("dense_decode_fwd")
assert dense_decode[:5] == ["q", "kcache", "head_size_v", "seqlens_k", "block_table"] and len(dense_decode) == 10, dense_decode
assert argument_names("permute_q_b_proj") == ["q_b_proj", "scale_factors", "h_q", "d_q"]
assert argument_names("permute_wv_proj") == ["wv_proj", "scale_factors", "wv_group_size", "d_o"]
print("OK -- sparse prefill, sparse decode, dense decode and the two weight permutations have the expected arguments")


## Example 3 — the interface's own argument checks refuse a wrong call before any kernel runs


In [ ]:
import torch, flash_mla

meta, placeholder = flash_mla.get_mla_metadata()
assert placeholder is None and not meta.have_initialized and meta.config is None, meta
q = torch.zeros(1, 1, 2, 576, dtype=torch.bfloat16)        # (batch, query length, heads, head size)
k_cache = torch.zeros(1, 64, 1, 656, dtype=torch.uint8)    # (blocks, block size, 1, bytes per token)
indices = torch.zeros(1, 1, 8, dtype=torch.int32)          # (batch, query length, top-k): sparse attention
refusals = {}
for label, call in {
    "metadata of the wrong type": lambda: flash_mla.flash_mla_with_kvcache(q, k_cache, None, None, 512, object()),
    "num_splits given": lambda: flash_mla.flash_mla_with_kvcache(q, k_cache, None, None, 512, meta, num_splits=1),
    "sparse with a causal mask": lambda: flash_mla.flash_mla_with_kvcache(
        q, k_cache, None, None, 512, flash_mla.get_mla_metadata()[0], causal=True, is_fp8_kvcache=True, indices=indices),
    "sparse without an FP8 cache": lambda: flash_mla.flash_mla_with_kvcache(
        q, k_cache, None, None, 512, flash_mla.get_mla_metadata()[0], indices=indices),
    "dense without a block table": lambda: flash_mla.flash_mla_with_kvcache(
        q, k_cache, None, None, 512, flash_mla.get_mla_metadata()[0]),
}.items():
    try:
        call()
        raise RuntimeError(f"{label}: the call was not refused")
    except AssertionError as error:
        refusals[label] = str(error).split(".")[0]
assert len(refusals) == 5 and "FlashMLASchedMeta" in refusals["metadata of the wrong type"], refusals
print("OK -- 5 wrong calls refused in Python, none reached a kernel:", sorted(refusals))


## Example 4 — on a Hopper (9.x) or Blackwell data-centre (10.x) GPU, sparse MLA attention equals the same attention computed by plain torch; on any other GPU FlashMLA refuses the call


In [ ]:
import torch, flash_mla
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
major, minor = torch.cuda.get_device_capability()
torch.manual_seed(0)
s_q, s_kv, heads, d_qk, d_v, topk = 4, 128, 64, 576, 512, 128
q = torch.randn(s_q, heads, d_qk, device="cuda", dtype=torch.bfloat16)
kv = torch.randn(s_kv, 1, d_qk, device="cuda", dtype=torch.bfloat16)
indices = torch.stack([torch.randperm(s_kv, device="cuda")[:topk] for _ in range(s_q)]).view(s_q, 1, topk).int()
scale = d_qk ** -0.5
if major not in (9, 10):
    try:
        flash_mla.flash_mla_sparse_fwd(q, kv, indices, scale, d_v)
        refusal = None
    except RuntimeError as error:
        refusal = str(error)
    assert refusal is not None and "only supported on SM90a and SM100f" in refusal, (
        f"FlashMLA no longer refuses its sparse attention on this {major}.{minor} GPU, which it has no kernels for; "
        f"expected its refusal 'only supported on SM90a and SM100f', got: "
        f"{'no refusal, the call RAN' if refusal is None else refusal[:400]}")
    print(f"SKIP: no attention result to compare on this {major}.{minor} GPU (FlashMLA has kernels for "
          f"9.x and 10.x only); OK -- FlashMLA refuses the call here, with its own message")
else:
    out, max_logits, lse = flash_mla.flash_mla_sparse_fwd(q, kv, indices, scale, d_v)
    gathered = kv.squeeze(1).float()[indices.squeeze(1).long()]  # (s_q, topk, d_qk): the keys each query attends to
    logits = (q.float() @ gathered.transpose(1, 2)) * scale      # (s_q, heads, topk)
    expected = torch.softmax(logits, dim=-1) @ gathered[..., :d_v]
    assert out.shape == (s_q, heads, d_v) and out.dtype == torch.bfloat16, (out.shape, out.dtype)
    error = (out.float() - expected).abs().max().item() / expected.abs().max().item()
    assert error < 3e-2, f"sparse attention differs from the torch reference: relative error {error:.3g}"
    assert torch.allclose(lse, torch.logsumexp(logits, dim=-1), atol=1e-2, rtol=1e-3), "log-sum-exp differs"
    assert torch.allclose(max_logits, logits.max(dim=-1).values, atol=1e-2, rtol=1e-3), "largest logits differ"
    print(f"OK -- sparse MLA attention on {torch.cuda.get_device_name(0)} matches torch: relative error {error:.2e}")